In [1]:
import seaborn as sns
import pandas as pd

In [2]:
healthexp = sns.load_dataset('healthexp')
healthexp

,Year,Country,Spending_USD,Life_Expectancy
0,1970,Germany,252.311,70.6
1,1970,France,192.143,72.2
2,1970,Great Britain,123.993,71.9
3,1970,Japan,150.437,72.0
4,1970,USA,326.961,70.9
...,...,...,...,...
269,2020,Germany,6938.983,81.1
270,2020,France,5468.418,82.3
271,2020,Great Britain,5018.700,80.4
272,2020,Japan,4665.641,84.7


In [3]:
healthexp = pd.get_dummies(healthexp, dtype=int)
healthexp

,Year,Spending_USD,Life_Expectancy,Country_Canada,Country_France,Country_Germany,Country_Great Britain,Country_Japan,Country_USA
0,1970,252.311,70.6,0,0,1,0,0,0
1,1970,192.143,72.2,0,1,0,0,0,0
2,1970,123.993,71.9,0,0,0,1,0,0
3,1970,150.437,72.0,0,0,0,0,1,0
4,1970,326.961,70.9,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...
269,2020,6938.983,81.1,0,0,1,0,0,0
270,2020,5468.418,82.3,0,1,0,0,0,0
271,2020,5018.700,80.4,0,0,0,1,0,0
272,2020,4665.641,84.7,0,0,0,0,1,0


In [4]:
X = healthexp.drop(['Life_Expectancy'], axis=1)
y = healthexp['Life_Expectancy']

In [5]:
from sklearn.model_selection import train_test_split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=54)

In [7]:
from sklearn.ensemble import RandomForestRegressor
rfr= RandomForestRegressor(random_state=34)

In [8]:
rfr.fit(X_train, y_train)

RandomForestRegressor(random_state=34)

In [9]:
y_pred = rfr.predict(X_test)

In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [11]:
mean_absolute_error(y_test, y_pred)

0.31138181818180044

In [12]:
mean_squared_error(y_test, y_pred)

0.1553235999999905

In [13]:
r2_score(y_test, y_pred)

0.9836234548107303

In [14]:
import optuna 
from sklearn.model_selection import cross_val_score

In [21]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 100,1000)
    max_depth = trial.suggest_int('max_depth', 10,50)
    min_samples_split = trial.suggest_int('min_samples_split', 2,32)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1,32)

    model = RandomForestRegressor(n_estimators=n_estimators,
                                 max_depth=max_depth,
                                 min_samples_split=min_samples_split,
                                 min_samples_leaf=min_samples_leaf)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error' ,n_jobs=-1).mean()
    return score

In [22]:
study = optuna.create_study(direction='maximize', sampler= optuna.samplers.RandomSampler(seed=42))

[I 2026-03-12 03:48:29,992] A new study created in memory with name: no-name-15e2bd32-093d-4662-b67c-63779d90d7c9


In [23]:
study.optimize(objective, n_trials=200)

[I 2026-03-12 03:48:36,365] Trial 0 finished with value: -2.309741991023285 and parameters: {'n_estimator': 437, 'max_depth': 48, 'min_samples_split': 24, 'min_samples_leaf': 20}. Best is trial 0 with value: -2.309741991023285.
[I 2026-03-12 03:48:37,500] Trial 1 finished with value: -2.9034630474207574 and parameters: {'n_estimator': 240, 'max_depth': 16, 'min_samples_split': 3, 'min_samples_leaf': 28}. Best is trial 0 with value: -2.309741991023285.
[I 2026-03-12 03:48:40,081] Trial 2 finished with value: -3.325201852407365 and parameters: {'n_estimator': 641, 'max_depth': 39, 'min_samples_split': 2, 'min_samples_leaf': 32}. Best is trial 0 with value: -2.309741991023285.
[I 2026-03-12 03:48:43,785] Trial 3 finished with value: -1.0133606571897464 and parameters: {'n_estimator': 850, 'max_depth': 18, 'min_samples_split': 7, 'min_samples_leaf': 6}. Best is trial 3 with value: -1.0133606571897464.
[I 2026-03-12 03:48:45,399] Trial 4 finished with value: -1.56398413329412 and parameters

In [24]:
study.best_params

{'n_estimator': 940,
 'max_depth': 45,
 'min_samples_split': 3,
 'min_samples_leaf': 1}

In [25]:
best_params = study.best_params

In [26]:
import matplotlib.pyplot as plt

In [28]:
optuna.visualization.plot_optimization_history(study)

In [29]:
optuna.visualization.plot_parallel_coordinate(study)

In [31]:
optuna.visualization.plot_slice(study, params=['n_estimator','max_depth','min_samples_split','min_samples_leaf'])

In [32]:
optuna.visualization.plot_param_importances(study)

In [35]:
best_n_estimators = best_params['n_estimator']
best_max_depth = best_params['max_depth']
best_min_samples_split = best_params['min_samples_split']
best_min_samples_leaf= best_params['min_samples_leaf']

In [36]:
best_model = RandomForestRegressor(n_estimators=best_n_estimators,
                                  max_depth=best_max_depth,
                                  min_samples_split=best_min_samples_split,
                                  min_samples_leaf=best_min_samples_leaf)

In [37]:
best_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=45, min_samples_split=3, n_estimators=940)

In [38]:
y_pred = best_model.predict(X_test)

In [39]:
mean_absolute_error(y_test, y_pred)

0.3239366038806912

In [40]:
mean_squared_error(y_test, y_pred)

0.16482445773479967

In [41]:
r2_score(y_test, y_pred)

0.9826217317884018